# ABC / XYZ Segmentation - H&M

Este notebook realiza la segmentación ABC (por valor) y XYZ (por variabilidad) de clientes usando datos de transacciones.
El objetivo es identificar los clientes más valiosos y entender la estabilidad de sus compras para estrategias personalizadas.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, stddev, count, year, month
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

spark = SparkSession.builder.appName("ABC_XYZ_HYM").getOrCreate()

## Carga y exploración inicial de datos

In [3]:
df = spark.read.parquet('../merge_pyspark')
print(f"Total de registros: {df.count():,}")
df.printSchema()
df.show(5)

Total de registros: 31,788,324
root
 |-- customer_id: string (nullable = true)
 |-- article_id: long (nullable = true)
 |-- Fecha: date (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: long (nullable = true)
 |-- product_code: integer (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- product_type_no: integer (nullable = true)
 |-- product_type_name: string (nullable = true)
 |-- product_group_name: string (nullable = true)
 |-- graphical_appearance_no: integer (nullable = true)
 |-- graphical_appearance_name: string (nullable = true)
 |-- colour_group_code: integer (nullable = true)
 |-- colour_group_name: string (nullable = true)
 |-- perceived_colour_value_id: integer (nullable = true)
 |-- perceived_colour_value_name: string (nullable = true)
 |-- perceived_colour_master_id: integer (nullable = true)
 |-- perceived_colour_master_name: string (nullable = true)
 |-- department_no: integer (nullable = true)
 |-- department_name: string (nullab

## Limpieza y preparación

In [4]:
df_clean = df.select('customer_id', 'Fecha', 'price') \
    .where(col('customer_id').isNotNull() & col('Fecha').isNotNull() & (col('price') > 0))
print(f"Registros limpios: {df_clean.count():,}")

Registros limpios: 31,788,324


## Métricas por cliente: total gastado, número de compras y variabilidad mensual

In [5]:
clientes = df_clean.groupBy('customer_id') \
    .agg(
        spark_sum('price').alias('total_gastado'),
        count('price').alias('num_compras')
    )

gasto_mensual = df_clean.withColumn('año', year('Fecha')).withColumn('mes', month('Fecha')) \
    .groupBy('customer_id', 'año', 'mes') \
    .agg(spark_sum('price').alias('gasto_mes'))

variabilidad = gasto_mensual.groupBy('customer_id') \
    .agg(stddev('gasto_mes').alias('std_gasto_mes'))

clientes = clientes.join(variabilidad, on='customer_id', how='left')

## Segmentación ABC (por valor)

- A: Top 5% de clientes por gasto total (más valiosos)
- B: Siguiente 15%
- C: 80% restantes

In [6]:
clientes_pd = clientes.toPandas()
clientes_pd['ABC'] = pd.qcut(clientes_pd['total_gastado'], q=[0, .8, .95, 1], labels=['C', 'B', 'A'])

## Segmentación XYZ (por variabilidad)

- X: Clientes con gasto mensual más estable (menor desviación estándar)
- Y: Variabilidad intermedia
- Z: Clientes con gasto más variable

In [ ]:
# Segmentación XYZ (por variabilidad)
std_values = clientes_pd['std_gasto_mes'].fillna(0)
bins = pd.qcut(std_values, q=[0, .33, .66, 1], retbins=True, duplicates='drop')[1]
num_bins = len(bins) - 1

if num_bins == 3:
    etiquetas = ['Z', 'Y', 'X']
elif num_bins == 2:
    etiquetas = ['Z', 'X']
else:
    etiquetas = ['X']

clientes_pd['XYZ'] = pd.cut(std_values, bins=bins, labels=etiquetas, include_lowest=True)

TypeError: qcut() got an unexpected keyword argument 'include_lowest'

## Visualización de segmentos ABC y XYZ

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(x='ABC', data=clientes_pd, palette='Set2')
plt.title('Distribución de clientes por segmento ABC')
plt.xlabel('Segmento ABC')
plt.ylabel('Número de clientes')
plt.show()

plt.figure(figsize=(8,6))
sns.countplot(x='XYZ', data=clientes_pd, palette='Set1')
plt.title('Distribución de clientes por segmento XYZ')
plt.xlabel('Segmento XYZ')
plt.ylabel('Número de clientes')
plt.show()

## Matriz combinada ABC/XYZ

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(pd.crosstab(clientes_pd['ABC'], clientes_pd['XYZ']), annot=True, fmt='d', cmap='Blues')
plt.title('Matriz ABC/XYZ de clientes')
plt.xlabel('Segmento XYZ')
plt.ylabel('Segmento ABC')
plt.show()

## Análisis profesional y recomendaciones

In [ ]:
from IPython.display import Markdown
Markdown("""
**Análisis profesional:**

- Segmento A: Clientes que más aportan al negocio, deben ser prioridad en retención y recompensas.
- Segmento X: Clientes con compras estables, ideales para predicción y programas de lealtad.
- Segmento AX: Clientes más valiosos y predecibles, foco de campañas VIP.
- Segmentos CZ y BY: Clientes menos prioritarios o con comportamiento variable, útiles para campañas de activación.

**Recomendaciones:**
- Personalizar ofertas y comunicaciones para clientes AX.
- Incentivar la recurrencia en segmentos B y C.
- Analizar causas de variabilidad en segmentos Y y Z para mejorar la estabilidad.
- Monitorear la evolución de los segmentos para ajustar estrategias de marketing.
""")

## Exportación de resultados y cierre de Spark

In [ ]:
clientes_pd.to_csv('segmentacion_abc_xyz_hym.csv', index=False)
print('Segmentación exportada a segmentacion_abc_xyz_hym.csv')

spark.stop()